In [1]:

import pandas as pd
import numpy as np
import os

# ==========================================
# RAILPULSE AI - FEATURE ENGINEERING
# ==========================================

input_path = "../data/processed/railpulse_next_station_pairs.csv"

output_path = "../data/processed/railpulse_model_features.csv"

os.makedirs("../data/processed", exist_ok=True)

# Features and metadata required
required_columns = [
    "train",
    "date",
    "current_station",
    "next_station",
    "current_arr_delay",
    "current_scheduled_arrival",
    "current_sequence",
    "current_distance",
    "distance_to_next",
    "next_arr_delay"
]

# Counters for validation
total_rows = 0
invalid_dates = 0
invalid_times = 0
missing_values = 0

if os.path.exists(output_path):
    os.remove(output_path)

first_chunk = True

for chunk in pd.read_csv(
    input_path,
    chunksize=100_000,
    usecols=required_columns,
    dtype={
        "train": str,
        "current_station": str,
        "next_station": str
    }
):

    total_rows += len(chunk)

    # --------------------------------------
    # 1. Convert numeric columns
    # --------------------------------------

    numeric_columns = [
        "current_arr_delay",
        "current_sequence",
        "current_distance",
        "distance_to_next",
        "next_arr_delay"
    ]

    for col in numeric_columns:
        chunk[col] = pd.to_numeric(
            chunk[col],
            errors="coerce"
        )

    # --------------------------------------
    # 2. Date-based features
    # --------------------------------------

    date = pd.to_datetime(
        chunk["date"],
        format="%Y-%m-%d",
        errors="coerce"
    )

    invalid_dates += date.isna().sum()

    day_of_week = date.dt.dayofweek

    # Cyclical weekly representation
    chunk["day_of_week_sin"] = np.sin(
        2 * np.pi * day_of_week / 7
    )

    chunk["day_of_week_cos"] = np.cos(
        2 * np.pi * day_of_week / 7
    )

    chunk["is_weekend"] = (
        day_of_week >= 5
    ).astype("int8")

    # --------------------------------------
    # 3. Scheduled arrival time features
    # --------------------------------------

    scheduled_time = pd.to_datetime(
        chunk["current_scheduled_arrival"],
        format="%I:%M %p",
        errors="coerce"
    )

    invalid_times += scheduled_time.isna().sum()

    minute_of_day = (
        scheduled_time.dt.hour * 60 +
        scheduled_time.dt.minute
    )

    chunk["arrival_hour_sin"] = np.sin(
        2 * np.pi * minute_of_day / 1440
    )

    chunk["arrival_hour_cos"] = np.cos(
        2 * np.pi * minute_of_day / 1440
    )

    # --------------------------------------
    # 4. Select model features and metadata
    # --------------------------------------

    final_columns = [
        "train",
        "date",
        "current_station",
        "next_station",

        "current_arr_delay",
        "distance_to_next",
        "current_distance",
        "current_sequence",

        "arrival_hour_sin",
        "arrival_hour_cos",

        "day_of_week_sin",
        "day_of_week_cos",
        "is_weekend",

        "next_arr_delay"
    ]

    result = chunk[final_columns].copy()

    missing_values += result.isna().sum().sum()

    # --------------------------------------
    # 5. Save chunk
    # --------------------------------------

    result.to_csv(
        output_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

    print("Processed rows:", total_rows)

# ==========================================
# FINAL VALIDATION
# ==========================================

print("\n" + "=" * 65)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 65)

print("Total processed rows:", total_rows)
print("Invalid dates:", invalid_dates)
print("Invalid scheduled times:", invalid_times)
print("Missing feature/target values:", missing_values)

print("\nSaved dataset:", output_path)

print("\nFEATURE ENGINEERING COMPLETED")


Processed rows: 100000
Processed rows: 200000
Processed rows: 300000
Processed rows: 400000
Processed rows: 500000
Processed rows: 600000
Processed rows: 700000
Processed rows: 800000
Processed rows: 900000
Processed rows: 1000000
Processed rows: 1100000
Processed rows: 1101443

FEATURE ENGINEERING SUMMARY
Total processed rows: 1101443
Invalid dates: 0
Invalid scheduled times: 0
Missing feature/target values: 0

Saved dataset: ../data/processed/railpulse_model_features.csv

FEATURE ENGINEERING COMPLETED
